# Complexity-04 — Décider sans connaître la suite

**Navigation** : [Index](README.md) | [Version Python](Complexity-04-OnlineAlgorithms-Python.ipynb) | [03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb) | [04b — Secrétaire matroïdal et k-server](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb)

> **Public : Licence.** Prérequis C# : bases de tableaux, fonctions locales et dictionnaires ; exécution avec le noyau .NET Interactive (`.net-csharp`). Aucune bibliothèque externe requise. Les conjectures de septembre 2026, le secrétaire matroïdal et le work function algorithm sont conservés dans l'approfondissement 04b ; ce notebook s'arrête aux trois objets classiques du domaine.

## Ce que ce notebook suppose

- [01 — Compter des pas](Complexity-01-StepCounting-Python.ipynb) : le coût d'un algorithme se **compte**, il ne se chronomètre pas ;
- [03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb) : comparer un algorithme à ceux d'une **classe entière**, et pourquoi une table finie ne démontre rien.

**Objectifs** : poser le modèle d'une décision **irréversible** prise sans connaître la suite ; mesurer la compétitivité des skis et de la pagination, puis la probabilité de choisir la meilleure personne au secrétaire sous ordre aléatoire ; séparer ce qui se mesure de ce qui se démontre. **Durée estimée** : 45 minutes.


## 1. Louer ou acheter, sans savoir combien de temps

Un magasin loue des skis 1 € par jour, ou les vend $B$ € une fois pour toutes. Vous skiez demain, mais vous ne savez pas jusqu'à quand : acheter trop tôt et acheter trop tard se paient tous les deux. Un algorithme **en ligne** décide jour après jour sans connaître la durée ; un algorithme **hors ligne** la connaît, et paie l'optimum $\min(n, B)$.

La règle en ligne la plus simple fixe un **seuil** : louer jusqu'au jour $T-1$, acheter au jour $T$. Comme la durée réelle reste inconnue, on ne juge pas la règle sur une saison mais sur **toutes** les durées possibles, par le pire rapport coût payé sur coût optimal — le **rapport de compétitivité**.

Reste à choisir $T$. Balayons-les tous, pour $B = 10$.


In [1]:
using System.Globalization;
using System.Linq;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime .NET {Environment.Version}; culture {CultureInfo.CurrentCulture.Name}");
int B = 10;
int CoutSeuil(int T, int n, int b) => n < T ? n : T - 1 + b;
int Optimum(int n, int b) => Math.Min(n, b);
Console.WriteLine($"Location de skis — achat B = {B} €, location 1 €/jour");
Console.WriteLine($"{"T",4} | {"pire rapport",12}");
for (int t=1;t<=B;t++) { double pire=Enumerable.Range(1,4*B).Max(n=>(double)CoutSeuil(t,n,B)/Optimum(n,B)); Console.WriteLine($"{t,4} | {pire,12:F4}"); }
double pireB=Enumerable.Range(1,4*B).Max(n=>(double)CoutSeuil(B,n,B)/Optimum(n,B));
if (Math.Abs(pireB-(2-1.0/B))>1e-12) throw new Exception("Contrôle du seuil B échoué");
Console.WriteLine($"\nMeilleur seuil T={B}, pire rapport={pireB:F4} = 2-1/{B}");

Runtime .NET 10.0.14; culture 


Location de skis — achat B = 10 €, location 1 €/jour


   T | pire rapport


   1 |      10.0000


   2 |       5.5000


   3 |       4.0000


... (4 outputs left out)

   8 |       2.1250


   9 |       2.0000


  10 |       1.9000



Meilleur seuil T=10, pire rapport=1.9000 = 2-1/10


**Lecture du résultat.** Le balayage désigne le seuil $T = B$ : louer les $B-1$ premiers jours, puis acheter au jour $B$. Son pire rapport vaut exactement $2 - 1/B$, soit $1{,}9$ pour $B = 10$ ; le pire cas est la saison de $n = B$ jours, où l'optimum paie $B$ et la règle paie $(B-1) + B$. Acheter plus tôt est bien pire — à $T = 1$ le pire rapport vaut $B$ entier. Aucun seuil de cette famille ne descend sous $2 - 1/B$, et il se trouve que la barrière est plus haute que la famille : elle vaut pour **tous** les algorithmes déterministes (§5).

## Exercices

Les trois exercices s'exécutent sans erreur sur des stubs (règle C.1 du dépôt) — le notebook tourne de bout en bout, corrigés compris dans le travail de l'étudiante et de l'étudiant.

### Exercice 1 — Le seuil optimal tient-il pour $B = 20$ ?

Réutilisez les fonctions C# `CoutSeuil(T, n, b)` et `Optimum(n, b)` avec $B=20$, et balayez les seuils de 1 à 20. Le minimum du pire rapport est-il encore atteint en $T = B$, et vaut-il $2 - 1/20$ ?

- **Indice :** la réponse est un tableau ordonné des vingt pires rapports, après balayage des durées.
- **Etape 1 :** balayer $T$ de 1 à 20 et afficher le pire rapport de chacun.
- **Etape 2 :** vérifier le minimum trouvé et le comparer à $2 - 1/B$.


In [2]:
// TODO étudiant : calculer les 20 pires rapports du balayage B=20.
double?[] pireRatioB20 = null;
bool ValiderB20(double?[] reponse) {
 if(reponse is null || reponse.Length!=20 || reponse.Any(x=>x is null || !double.IsFinite(x.Value))) return false;
 for(int t=1;t<=20;t++){double attendu=Enumerable.Range(1,80).Max(n=>(double)CoutSeuil(t,n,20)/Optimum(n,20));if(Math.Abs(reponse[t-1]!.Value-attendu)>1e-10)return false;}
 return Math.Abs(reponse.Min(x=>x!.Value)-1.95)<1e-12;
}
Console.WriteLine(ValiderB20(pireRatioB20)?"Réponse B=20 vérifiée sur les 20 seuils.":"Exercice à compléter : calculer le pire rapport de chaque seuil T=1..20 pour B=20.");

Exercice à compléter : calculer le pire rapport de chaque seuil T=1..20 pour B=20.


## 2. Tirer au sort le jour de l'achat

Une règle déterministe a un défaut que le tableau rend visible : son pire cas est **une** durée, et l'adversaire la choisit. Randomiser change la nature du jeu — l'adversaire ne choisit plus contre une règle, mais contre une **distribution**. On tire le jour d'achat selon $p_T$ proportionnelle à $(1 + 1/B)^{T-1}$ : une distribution qui privilégie les achats tardifs sans jamais exclure les précoces.

Le rapport devient alors une **espérance**, $\mathbb{E}[\mathrm{ALG}(n)] / \mathrm{OPT}(n)$, maximisée sur les durées. Comparons-la au seuil déterministe et à la valeur que la littérature donne pour optimale, $e/(e-1) \approx 1{,}582$.


In [3]:
double PireRatioRandomise(int b) {
 var p=Enumerable.Range(1,b).Select(t=>Math.Pow(1+1.0/b,t-1)).ToArray(); double z=p.Sum(); for(int i=0;i<p.Length;i++) p[i]/=z;
 var ratios=new List<double>(); double cp=0, cm=0;
 for(int n=1;n<b;n++){cp+=p[n-1]; cm+=p[n-1]*(n-1+b); ratios.Add((n*(1-cp)+cm)/n);}
 ratios.Add((p.Select((x,i)=>x*(i+1)).Sum()-1+b)/b); return ratios.Max();
}
Console.WriteLine($"{"B",6} | {"randomisé",10} | {"déterministe",12} | {"e/(e-1)",9}");
foreach(int b in new[]{10,100,1000,10000}) Console.WriteLine($"{b,6} | {PireRatioRandomise(b),10:F5} | {2-1.0/b,12:F5} | {Math.E/(Math.E-1),9:F5}");
Console.WriteLine("Espérances calculées par somme exacte des probabilités discrètes, sans simulation.");

     B |  randomisé | déterministe |   e/(e-1)


    10 |    1.56471 |      1.90000 |   1.58198


   100 |    1.58071 |      1.99000 |   1.58198


  1000 |    1.58185 |      1.99900 |   1.58198


 10000 |    1.58196 |      1.99990 |   1.58198


Espérances calculées par somme exacte des probabilités discrètes, sans simulation.


**Lecture du résultat.** La randomisation descend loin sous la barrière déterministe : le pire rapport mesuré vaut $1{,}56471$ pour $B = 10$, puis $1{,}58071$, $1{,}58185$ et $1{,}58196$ — il **converge vers $e/(e-1) \approx 1{,}58198$**, la valeur asymptotique optimale citée dans la littérature. Cette valeur n'a pas été recopiée : elle sort de la mesure, sur une distribution construite ici. À $B=10$, le rapport déterministe affiché est $1{,}90000$ : l’écart avec $1{,}56471$ vaut $0{,}33529$. À $B=10\,000$, les rapports sont $1{,}99990$ et $1{,}58196$, écart $0{,}41794$. Quand $B$ grandit, l’écart tend vers $2-e/(e-1)\approx0{,}41802$. Ces limites asymptotiques ne sont ni une borne inférieure pour chaque ratio discret fini, ni une preuve d’optimalité de cette distribution à chaque $B$.

## 3. Garder $k$ pages quand on ne sait pas lesquelles reviendront

Une mémoire rapide de $k$ cases, des pages demandées une par une ; une page absente coûte un **défaut** et chasse une page présente. Le futur est inconnu : c'est le problème de la **pagination**, et c'est le dilemme des skis sous une autre forme — garder une page (payer plus tard) ou la remplacer (payer maintenant).

Deux algorithmes, deux philosophies :

- **Belady (hors ligne)** : évincer la page dont le **prochain usage est le plus lointain**. Il connaît la suite ; c'est l'optimum hors ligne, pas une stratégie applicable telle quelle ;
- **LRU (en ligne)** : évincer la page **utilisée le plus anciennement**. Aucune connaissance du futur.

Mesurons-les sur un motif qu'un adversaire peut construire — $k+1$ pages demandées en cycle, de sorte que la page demandée soit toujours celle qui vient d'être évincée — puis sur des requêtes aléatoires, où le futur n'est pas hostile.


In [4]:
int Belady(int[] r,int k){var cache=new HashSet<int>();int miss=0;for(int t=0;t<r.Length;t++){if(cache.Contains(r[t]))continue;miss++;if(cache.Count<k){cache.Add(r[t]);continue;}int victim=cache.First(); int far=-1;foreach(int p in cache){int next=Array.IndexOf(r,p,t+1);if(next<0)next=int.MaxValue;if(next>far){far=next;victim=p;}} cache.Remove(victim);cache.Add(r[t]);}return miss;}
int Lru(int[] r,int k){var c=new List<int>();int m=0;foreach(int x in r){if(c.Remove(x)){c.Add(x);continue;}m++;if(c.Count==k)c.RemoveAt(0);c.Add(x);}return m;}
int Fifo(int[] r,int k){var c=new List<int>();int m=0;foreach(int x in r){if(c.Contains(x))continue;m++;if(c.Count==k)c.RemoveAt(0);c.Add(x);}return m;}
Console.WriteLine("Motif cyclique sur k+1 pages");Console.WriteLine(" k |   T |   OPT |  LRU | FIFO | rapport LRU | rapport FIFO");
var ratiosCycliques=new Dictionary<(int,int),double>();
foreach(int k in new[]{2,3,4,5})foreach(int t in new[]{60,600}){var r=Enumerable.Range(0,t).Select(i=>i%(k+1)).ToArray();int o=Belady(r,k),l=Lru(r,k),f=Fifo(r,k);ratiosCycliques[(k,t)]=(double)l/o;Console.WriteLine($"{k,2} | {t,4} | {o,5} | {l,4} | {f,4} | {(double)l/o,11:F3} | {(double)f/o,12:F3}");}
var rng=new Random(42);int[] reqAlea=Enumerable.Range(0,600).Select(_=>rng.Next(20)).ToArray();int o5=Belady(reqAlea,5),l5=Lru(reqAlea,5),f5=Fifo(reqAlea,5);double ratioLruAlea=(double)l5/o5,ratioFifoAlea=(double)f5/o5;Console.WriteLine($"Aléatoire: OPT={o5}, LRU={l5}, FIFO={f5}; rapports {ratioLruAlea:F4}/{ratioFifoAlea:F4}");

Motif cyclique sur k+1 pages


 k |   T |   OPT |  LRU | FIFO | rapport LRU | rapport FIFO


 2 |   60 |    31 |   60 |   60 |       1.935 |        1.935


 2 |  600 |   301 |  600 |  600 |       1.993 |        1.993


 3 |   60 |    22 |   60 |   60 |       2.727 |        2.727


 3 |  600 |   202 |  600 |  600 |       2.970 |        2.970


 4 |   60 |    18 |   60 |   60 |       3.333 |        3.333


 4 |  600 |   153 |  600 |  600 |       3.922 |        3.922


 5 |   60 |    16 |   60 |   60 |       3.750 |        3.750


 5 |  600 |   124 |  600 |  600 |       4.839 |        4.839


Aléatoire: OPT=292, LRU=448, FIFO=446; rapports 1.5342/1.5274


**Lecture du résultat.** Sur le motif cyclique, LRU fait un défaut à **chaque** requête, et son rapport mesuré tend vers $k$ : $1{,}99$ pour $k = 2$, $2{,}97$ pour $k = 3$, $3{,}92$ pour $k = 4$, $4{,}84$ pour $k = 5$ — la limite $k$ est atteinte de plus près à mesure que la séquence s'allonge. FIFO, dont la règle d'éviction est pourtant différente, affiche **exactement** les mêmes rapports sur ce motif : ce qui coûte ici n'est pas la règle d'éviction, c'est l'absence de futur. Sur les requêtes aléatoires, le LRU mesuré vaut $1{,}5342$ : le pire cas est **adversarial**, pas moyen — la même leçon que pour les skis.

### Exercice 2 — Une troisième politique d'éviction

Ajoutez une politique qui évince la page **la moins fréquemment demandée depuis le début** (LFU), avec la signature C# `int? Lfu(int[] requetes, int k)`, à comparer à `Lru(int[] r, int k)`, puis mesurez son rapport en regard de celui de LRU sur les deux motifs : le cyclique ($k+1$ pages) et l'aléatoire (20 pages).

- **Indice :** tenir un compteur d'occurrences par page, et évincer celle dont le compteur est le plus petit.
- **Etape 1 :** implémenter la signature `int? Lfu(int[] requetes, int k)` et comparer avec `Lru(int[] r, int k)`.
- **Etape 2 :** imprimer son rapport et celui de LRU sur les deux motifs.


In [5]:
// TODO étudiant : implémenter LFU avec fréquences globales et tie-break FIFO (âge de la dernière entrée dans le cache).
int? Lfu(int[] requetes,int k) { return null; }
var cycleLfu=Enumerable.Range(0,60).Select(i=>i%3).ToArray();
int? lfuCycle=Lfu(cycleLfu,2), lfuAlea=Lfu(reqAlea,5);
int? lfuFixture1=Lfu(new[]{1,2,1,3,4,1,2,5},2);
int? lfuFixture2=Lfu(new[]{1,2,3,1,2,4,1,2,3,4,5,1},3);
bool fixturesLFU=lfuFixture1==6 && lfuFixture2==7;
bool mesuresLFU=lfuCycle.HasValue && lfuAlea.HasValue && lfuCycle.Value>=Belady(cycleLfu,2) && lfuCycle.Value<=cycleLfu.Length && lfuAlea.Value>=o5 && lfuAlea.Value<=reqAlea.Length;
if(fixturesLFU && mesuresLFU){
 Console.WriteLine($"LFU cycle: {lfuCycle.Value} défauts / {Belady(cycleLfu,2)} OPT; rapport {((double)lfuCycle.Value/Belady(cycleLfu,2)):F4}");
 Console.WriteLine($"LRU cycle: {Lru(cycleLfu,2)} défauts; comparaison mesurée");
 Console.WriteLine($"LFU aléatoire: {lfuAlea.Value} défauts / {o5} OPT; rapport {((double)lfuAlea.Value/o5):F4}");
 Console.WriteLine($"LRU aléatoire: {l5} défauts / {o5} OPT; comparaison mesurée");
 Console.WriteLine("Réponse LFU validée par fixtures 6/7 et bornes OPT ≤ défauts ≤ longueur.");
}else Console.WriteLine("Exercice à compléter : aucun résultat LFU affiché tant que les deux fixtures (6/7) et les bornes des deux motifs ne valident pas.");

Exercice à compléter : aucun résultat LFU affiché tant que les deux fixtures (6/7) et les bornes des deux motifs ne valident pas.


## 4. Choisir sans jamais pouvoir revenir en arrière

Dernier problème, le plus pur des trois : $n$ candidates et candidats se présentent **un par un**, dans un ordre aléatoire. À chaque entretien on n'apprend que le **rang** de la personne parmi celles déjà vues, et il faut décider **immédiatement** : accepter, ou laisser passer pour toujours. Objectif : maximiser la probabilité d'accepter la meilleure personne.

La règle classique ne regarde aucune valeur, seulement l'**ordre** : observer sans accepter une fraction $\gamma$ des arrivées, puis accepter la première personne qui dépasse tout ce qui a été vu. Pour $n$ grand, le seuil optimal tend vers $\gamma=1/e$ et la probabilité optimale tend vers $1/e$. À taille finie, le maximum exact peut différer de $1/e$ ; cette expérience en estime le succès sans le démontrer. Balayons $\gamma$ : la courbe doit le montrer.


In [6]:
bool Secretaire(int[] perm,double gamma){int n=perm.Length, seuil=Math.Max(1,(int)(gamma*n));int vu=perm.Take(seuil).Max();foreach(int x in perm.Skip(seuil))if(x>vu)return x==n-1;return false;}
long permutationsReelles=0;
double TauxSucces(int n,double gamma,int runs,Random rng){int wins=0;var p=Enumerable.Range(0,n).ToArray();for(int run=0;run<runs;run++){for(int i=n-1;i>0;i--){int j=rng.Next(i+1);(p[i],p[j])=(p[j],p[i]);}if(Secretaire(p,gamma))wins++;permutationsReelles++;}return (double)wins/runs;}
var rngSec=new Random(42);Console.WriteLine($"{"gamma",7} | {"n=100",8} | {"n=1000",8}");var taux=new Dictionary<double,(double,double)>();
foreach(double g in new[]{.1,.2,.3,1/Math.E,.4,.5,.6,.8}){double a=TauxSucces(100,g,20000,rngSec),b=TauxSucces(1000,g,20000,rngSec);taux[g]=(a,b);Console.WriteLine($"{g,7:F4} | {a,8:F4} | {b,8:F4}");}
double sigma=Math.Sqrt(.368*.632/20000);Console.WriteLine($"\n1/e={1/Math.E:F4}; e/(e-1)={Math.E/(Math.E-1):F4}; écart-type estimé={sigma:F4}; permutations réelles={permutationsReelles:N0}; seed=42");

  gamma |    n=100 |   n=1000


 0.1000 |   0.2359 |   0.2324


 0.2000 |   0.3234 |   0.3206


 0.3000 |   0.3718 |   0.3534


 0.3679 |   0.3646 |   0.3713


 0.4000 |   0.3655 |   0.3659


 0.5000 |   0.3456 |   0.3459


 0.6000 |   0.3057 |   0.3033


 0.8000 |   0.1806 |   0.1771



1/e=0.3679; e/(e-1)=1.5820; écart-type estimé=0.0034; permutations réelles=320,000; seed=42


**Lecture du résultat.** Les mesures fluctuent autour de la zone théorique $1/e$. Pour $n=100$, les succès à $\gamma=1/e$ et $0{,}4$ valent respectivement $0{,}3646$ et $0{,}3655$, écart inférieur à l’erreur-type estimée $0{,}0034$ : ce banc ne les départage pas. À $n=1000$, ces estimations sont $0{,}3713$ et $0{,}3659$ ; le bruit reste visible. La grille fine proposée en exercice augmente la résolution des seuils, sans transformer une estimation Monte Carlo en preuve. En s’écartant franchement du seuil, le succès baisse : observer 10 % des arrivées donne $0{,}2324$ et 80 % donne $0{,}1771$ pour $n=1000$.

### Exercice 3 — Le sommet est-il vraiment à $1/e$ ?

Mesurez $\gamma=0{,}30,0{,}31,\ldots,0{,}45$ dans cet ordre, pour $n=1000$ et au moins 20 000 permutations par point. Créez un seul `Random(42)` au début du balayage et réutilisez-le pour tous les points ; la fonction C# est `TauxSucces(n, gamma, runs, rng)`. La garde rejoue chaque point avec le nombre de runs déclaré et ce même protocole. Où tombe le sommet estimé, et que permet d’en conclure cette résolution ?

- **Indice :** l'écart-type d'une proportion $p$ sur $N$ tirages vaut $\sqrt{p(1-p)/N}$ — ici environ $0{,}003$.
- **Etape 1 :** calculer la courbe sur la grille fine avec `TauxSucces(1000, gamma, runs, rngFin)`.
- **Etape 2 :** renseigner les clés exactes `0.30` à `0.45` et, pour chacune, le couple `(succes, runs)`. La validation rejoue le Monte Carlo avec le nombre de runs fourni (au moins 20 000), partageant un seul `Random(42)` entre les points.


In [7]:
var gammaFin=Enumerable.Range(30,16).Select(i=>i/100.0).ToArray();
// TODO étudiant : un RNG Random(42) partagé, puis un balayage dans l'ordre croissant.
Dictionary<double,(double succes,int runs)> sommetFin=null;
bool grilleFineComplete=false;
if(sommetFin is not null && sommetFin.Count==gammaFin.Length && sommetFin.Keys.Order().SequenceEqual(gammaFin) && sommetFin.All(kv=>double.IsFinite(kv.Value.succes)&&kv.Value.succes>=0&&kv.Value.succes<=1&&kv.Value.runs>=20000)) {
 var controleRng=new Random(42); grilleFineComplete=true;
 foreach(double g in gammaFin){var mesure=sommetFin[g];double attendu=TauxSucces(1000,g,mesure.runs,controleRng);if(Math.Abs(mesure.succes-attendu)>1e-12){grilleFineComplete=false;break;}}
}
Console.WriteLine(grilleFineComplete?"Grille secrétaire vérifiée : 16 seuils exacts, permutations rejouées au nombre déclaré et RNG partagé.":"Exercice à compléter : 16 seuils exacts, succès finis, runs ≥20000; sinon aucune mesure n'est certifiée.");

Exercice à compléter : 16 seuils exacts, succès finis, runs ≥20000; sinon aucune mesure n'est certifiée.


## 5. Ce qu'aucune de ces expériences ne démontre

Les trois sections précédentes sont des **tables finies** : elles évaluent des stratégies données sur des entrées choisies. Aucune ne peut établir qu'**aucune** stratégie ne fait mieux — exactement la limite que le notebook [03](Complexity-03-HierarchieTemporelle-Python.ipynb) avait posée pour les budgets de temps. Quatre énoncés le font à leur place ; ils sont **cités ici sans être démontrés** :

| Énoncé | Contenu | Ce que la mesure en montre |
|---|---|---|
| Skis, déterministe | aucun algorithme déterministe ne fait mieux que $2 - 1/B$ | notre meilleur seuil l'atteint : $1{,}9$ |
| Skis, randomisé | la référence théorique asymptotique tend vers $e/(e-1) \approx 1{,}582$ | la nôtre y converge : $1{,}58196$ |
| Pagination | le rapport optimal d'un algorithme déterministe est exactement $k$ (Sleator et Tarjan, 1985) | LRU l'atteint sur le motif cyclique |
| Secrétaire | la probabilité optimale tend vers $1/e$ dans la limite $n\to\infty$ | les essais dans la zone $\gamma\approx0{,}3$–$0{,}4$ donnent des succès élevés, mais la position du pic échantillonnal varie selon $n$ et le RNG |

Voici, en une table, tout ce que ce notebook mesure.


In [8]:
var resume=new[]{("Skis T=B, pire rapport",pireB.ToString("F4",CultureInfo.InvariantCulture),"référence déterministe 2-1/B"),("Skis randomisé B=10000",PireRatioRandomise(10000).ToString("F5",CultureInfo.InvariantCulture),"limite asymptotique e/(e-1)"),("Pagination cyclique k=5,T=600",ratiosCycliques[(5,600)].ToString("F3",CultureInfo.InvariantCulture),"borne de compétitivité k"),("Pagination aléatoire, LRU",ratioLruAlea.ToString("F4",CultureInfo.InvariantCulture),"pire cas adversarial"),("Secrétaire n=1000,gamma=1/e",taux[1/Math.E].Item2.ToString("F4",CultureInfo.InvariantCulture),"limite asymptotique 1/e")};
Console.WriteLine($"{"Mesure",40} {"valeur",8}   référence");foreach(var row in resume)Console.WriteLine($"{row.Item1,-40} {row.Item2,8}   {row.Item3}");

                                  Mesure   valeur   référence


Skis T=B, pire rapport                     1.9000   référence déterministe 2-1/B


Skis randomisé B=10000                    1.58196   limite asymptotique e/(e-1)


Pagination cyclique k=5,T=600               4.839   borne de compétitivité k


Pagination aléatoire, LRU                  1.5342   pire cas adversarial


Secrétaire n=1000,gamma=1/e                0.3713   limite asymptotique 1/e


## À retenir

1. **Le meilleur seuil des skis est d'acheter au jour B.** Pour $B = 10$, le balayage des seuils fait tomber le pire rapport de 10,0000 ($T = 1$) à **1,9000 $= 2 - 1/B$** ($T = 10$) — le minimum est atteint au seuil annoncé.
2. **La randomisation converge vers $e/(e-1)$.** À $B=10$, déterministe/randomisé valent **1,90000/1,56471** (écart **0,33529**) ; à $B=10\,000$, ils valent **1,99990/1,58196** (écart **0,41794**). L’écart tend vers $2-e/(e-1)\approx0{,}41802$ ; la mesure n’établit pas l’optimalité de la distribution à tout $B$ fini.
3. **LRU est exactement l'algorithme que l'adversaire veut.** Sur le motif cyclique des $k+1$ pages ($k = 5$, $T = 600$) : OPT = 124, LRU = 600 défauts, rapport **4,839** — la borne $k$ en vue ; sur le motif aléatoire (20 pages), le rapport retombe à **1,5342** (LRU) et **1,5274** (FIFO) : le pire cas est adversarial, pas le cas moyen.
4. **La probabilité de succès optimale tend vers $1/e$ quand $n$ grandit.** À $\gamma = 1/e$ : succès **0,3646** ($n = 100$) et **0,3713** ($n = 1000$), ces estimations Monte Carlo fluctuent autour de la limite $1/e$ ; à $n=100$, l’écart observé entre $\gamma=1/e$ et $0{,}4$ vaut $0{,}0009$, inférieur à l’erreur-type estimée $0{,}0034$. À taille finie, $1/e$ n’est pas nécessairement la probabilité optimale exacte.
5. **Ce que les mesures finies ne démontrent pas.** Les tableaux évaluent les stratégies testées : rapports des skis et de la pagination face à l’optimum hors ligne, et probabilité de choisir la meilleure personne sous permutation uniforme au secrétaire. Ils n’établissent pas la garantie randomisée pour chaque $B$ fini, la compétitivité de toute stratégie de pagination, ni le maximum exact du secrétaire à chaque $n$ ; les résultats cités sont asymptotiques ou portent sur leurs familles précises.

| Geste | Vérifié ici | Non démontré ici |
|---|---|---|
| Skis | le meilleur seuil et son pire rapport, sur toutes les durées | qu'aucun algorithme ne fasse mieux, ni en déterministe ni en randomisé |
| Pagination | LRU, FIFO et Belady mesurés sur deux motifs | que $k$ soit une borne pour **tout** algorithme en ligne |
| Secrétaire | le succès mesuré sous permutation uniforme | que l'optimum fini tende vers $1/e$ lorsque $n$ grandit |

Les trois problèmes partagent des décisions **irréversibles** prises sans connaître la suite, mais leurs critères diffèrent. Pour les skis et la pagination, on compare un coût au pire cas à l’optimum hors ligne ; pour le secrétaire, on mesure la probabilité de choisir la meilleure personne sous une permutation uniforme. C’est le même geste de méthode que dans [03](Complexity-03-HierarchieTemporelle-Python.ipynb) : des tables finies mesurent des cas, tandis que les garanties et limites relèvent des théorèmes cités.

L'approfondissement [04b — Secrétaire matroïdal et k-server](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb) reprend ces objets là où ils deviennent de la recherche : le work function algorithm pour $k$ serveurs, le secrétaire matroïdal, et deux conjectures de septembre 2026.

**Références**

- Belady, L. A. (1966), *A study of replacement algorithms for a virtual-storage computer*, IBM Systems Journal 5(2), 78–101.
- Sleator, D. D. et Tarjan, R. E. (1985), *Amortized efficiency of list update and paging rules*, Communications of the ACM 28(2), 202–208.
- Karlin, A. R., Manasse, M. S., McGeoch, L. A. et Owicki, S. (1994), *Competitive randomized algorithms for nonuniform problems*, Algorithmica 11(6), 542–571.
- Borodin, A. et El-Yaniv, R. (1998), *Online Computation and Competitive Analysis*, Cambridge University Press.
- Lindley, D. V. (1961), *Dynamic programming and decision theory*, Applied Statistics 10, 39–51 ; Dynkin, E. B. (1963), *The optimum choice of the instant for stopping a Markov process*, Soviet Mathematics 4, 627–629.


## Vérifiez votre compréhension

1. Pourquoi le pire rapport du seuil $T = B$ vaut-il exactement $2 - 1/B$ (mesuré : 1,9000 pour $B = 10$) ?
2. Quels sont les écarts mesurés à $B=10$ et $B=10\,000$, et vers quelle différence asymptotique tendent-ils ?
3. Pourquoi le rapport LRU explose-t-il à 4,839 sur le motif cyclique (OPT 124, LRU 600) et retombe-t-il à 1,5342 sur le motif aléatoire ?
4. La mesure à $n=100$ ne sépare pas les deux seuils : 0,3646 à $\gamma = 1/e$ et 0,3655 à $\gamma = 0{,}4$ ($n = 100$), pour un écart-type de 0,0034. La mesure peut-elle départager les deux réglages ?

<details><summary>Réponses</summary>

1. Deux régimes se raccordent : pour $n < B$ l'algorithme paie $n$ comme l'optimum (rapport 1) ; pour $n \geq B$ il a payé $B-1$ locations puis l'achat, $2B-1$ contre un optimum $B$. Le pire des deux régimes est $2 - 1/B$, atteint pour toute durée $n \geq B$.
2. À $B=10$, $1{,}90000-1{,}56471=0{,}33529$ ; à $B=10\,000$, $1{,}99990-1{,}58196=0{,}41794$. L’écart croît vers $2-e/(e-1)\approx0{,}41802$. Ce sont des limites de cette famille de stratégies, pas une preuve d’optimalité à chaque $B$ fini.
3. Le motif cyclique sur $k+1$ pages est l'adversaire exact : chaque page demandée vient d'être évincée, LRU faut à chaque requête (600 défauts sur 600 requêtes) tandis que Belady garde la page qui resservira. Le motif aléatoire n'est pas adversarial — la borne $k$ est un pire cas, pas une moyenne.
4. Non : à $n=100$, l'écart observé entre les deux seuils (0,0009) est inférieur à l'erreur-type estimée (0,0034). La mesure confirme le niveau $\approx 0{,}368$ aux deux réglages, sans pouvoir ordonner $1/e$ et 0,4.

</details>


## Pour aller plus loin

- **Carnet suivant** : [Complexity-04b — Secrétaire matroïdal et k-server](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb) — ces objets là où ils deviennent de la recherche : work function algorithm, matroïdes, deux conjectures de septembre 2026.
- **Dans la même série** : [Complexity-04c — k-server : work function](Complexity-04c-KServer-WorkFunction-Python.ipynb) — le WFA exact sur métriques jouets ; [Complexity-04d — Secrétaire matroïdal](Complexity-04d-Secretaire-Matroidal-Python.ipynb) — l'algorithme de Singla implémenté, garantie 1/4.
- **Carnet voisin d'une autre série** : [GameTheory-02 — Forme normale (Python)](../GameTheory/GameTheory-02-NormalForm-Python.ipynb) — l'adversaire du pire cas devient un joueur : stratégies pures, mixtes, minimax.
- **Lectures** : les références complètes (Belady 1966 ; Sleator et Tarjan 1985 ; Karlin, Manasse, McGeoch et Owicki 1994 ; Borodin et El-Yaniv 1998 ; Lindley 1961 / Dynkin 1963) sont citées à la fin du bloc « À retenir » ci-dessus.
